Copyright 2026, Battelle Energy Alliance, LLC, ALL RIGHTS RESERVED

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
os.chdir("../..")
from HelpingFunctions import ERCOTProcessor
from HelpingFunctions import WeatherProcessing
from HelpingFunctions import FeatureEngineering
from HelpingFunctions import ForecastingHelpers

import onnxruntime as ort
ort.set_default_logger_severity(4)

In [2]:
full_df = pd.read_csv('ElectricityDemandAustinTX/LoadForecastingAttacks/full_data.csv', parse_dates=['time'], index_col=['time']) 

In [3]:
hourly_res_norm = ForecastingHelpers.hourlyresiduals(full_df)

/home/ortild/Amaranth/opensourcegridmodeling/ElectricityDemandAustinTX/LoadForecastingAttacks/HelpingFunctions/ForecastingHelpers.py:74: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  hourly_res_norm['load'] = df_norm['load'].groupby(pd.Grouper(freq='M')).transform(lambda x: x - x.mean())


In [4]:
 # train-validate-test split
train = hourly_res_norm[:'2014']
validate = hourly_res_norm['2015':'2016']
test = hourly_res_norm['2017':]

# setup training variables 
exog_tr = train.iloc[:,1:].values
ar_tr = train['load'].shift().bfill().values[:,None]
X_tr = np.hstack([ar_tr, exog_tr])
y_tr = train['load'].values

# setup validation variables
exog_val = validate.iloc[:,1:].values
y_val = validate['load'].values

# setup testing variables
exog_te = test.iloc[:,1:].values
ar_test = test['load'].shift().bfill().values[:,None]
y_test = test['load'].values
X_test = np.hstack([ar_test, exog_te])

# setup miscellaneous variables
yp_full = hourly_res_norm.loc[:'2016','load']
yp_val = hourly_res_norm.loc['2015':'2016','load']
yp_te = hourly_res_norm.loc['2017':,'load']
y_init_val = np.hstack([y_tr[-1], validate.iloc[167::168,0].values])
y_init_te = np.hstack([y_val[-1], test.iloc[167::168,0].values])

mlp

In [5]:
import onnx
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx")

In [6]:
def forecast(session, exog, y_init, label_name, input_name):
    """given a trained model, exogenous features, and initial AR term, makes forecasting predictions"""
    yhat = []
    y_ci = []
    Xi_te = np.hstack([y_init, exog[0]])[None,:]
    for i in range(len(exog)-1):
        yhat_i = sess.run([label_name], {input_name: Xi_te.astype(np.float32)})[0][0]
        yhat.append(yhat_i)
        Xi_te = np.hstack([yhat_i, exog[i+1]])[None,:]
    yhat_i = sess.run([label_name], {input_name: Xi_te.astype(np.float32)})[0][0]
    yhat.append(yhat_i)
    return np.array(yhat)

def weekly_forecast(indexes, session, exog, y_init, label_name, input_name):
    """given a trained model exogenous features, and initial AR term, makes a series of 1-week-out forecasts"""
    yhat = []
    for i, yi in enumerate(y_init):
        exog_i = exog[168*i:168*(i+1),:]
        if exog_i.shape[0] < 1:
            break
        y_hat_i = forecast(session, exog_i, yi, label_name, input_name)
        yhat.append(y_hat_i)
    mapie_hat = pd.DataFrame(np.vstack(yhat).reshape(-1))
    return mapie_hat.values.ravel()

In [10]:
# Compute the prediction with onnxruntime.
import onnxruntime as rt

sess = rt.InferenceSession("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx", providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
label_name = sess.get_outputs()[0].name
print(sess.get_inputs()[0])
print(sess.get_outputs()[0])
#pred_onx = sess.run([label_name], {input_name: exog_te})[0]

NodeArg(name='X', type='tensor(float)', shape=[None, 13])
NodeArg(name='variable', type='tensor(float)', shape=[None, 1])


In [8]:
preds_onx = weekly_forecast(yp_te.index, sess, exog_te, y_init_te, label_name, input_name) 

In [9]:
 # plotting testing
print('MAE:', ForecastingHelpers.compute_mae(y_test, preds_onx))

MAE: 0.053691206107538136


Explore MAE Structures

In [12]:
import onnx
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/mlp_no_mapie.onnx")
graph = onnx_model.graph
print(onnx_model)
# Extract model weights and intercepts and set as variables
weights = onnx_model.graph.initializer[0].ListFields()[3][1][:]
intercept = onnx_model.graph.initializer[1].ListFields()[3][1][:]

print(intercept)

ir_version: 10
producer_name: "skl2onnx"
producer_version: "1.19.1"
domain: "ai.onnx"
model_version: 0
doc_string: ""
graph {
  node {
    input: "X"
    output: "cast_input"
    name: "Cast"
    op_type: "Cast"
    attribute {
      name: "to"
      i: 1
      type: INT
    }
    domain: ""
  }
  node {
    input: "cast_input"
    input: "coefficient"
    output: "mul_result"
    name: "MatMul"
    op_type: "MatMul"
    domain: ""
  }
  node {
    input: "mul_result"
    input: "intercepts"
    output: "add_result"
    name: "Add"
    op_type: "Add"
    domain: ""
  }
  node {
    input: "add_result"
    output: "next_activations"
    name: "Relu"
    op_type: "Relu"
    domain: ""
  }
  node {
    input: "next_activations"
    input: "coefficient1"
    output: "mul_result1"
    name: "MatMul1"
    op_type: "MatMul"
    domain: ""
  }
  node {
    input: "mul_result1"
    input: "intercepts1"
    output: "add_result1"
    name: "Add1"
    op_type: "Add"
    domain: ""
  }
  node {
   

In [ ]:
import onnx

def get_tree_ensemble_attributes(model_path):
    """Loads an ONNX model and extracts attributes from a TreeEnsembleRegressor node."""
    model = onnx.load(model_path)
    nodesInfo = {}
    for node in model.graph.node:
        if node.op_type == "TreeEnsembleRegressor":
            print(f"Found TreeEnsembleRegressor node: {node.name}")

            # The attributes contain the tree structure
            for attr in node.attribute:
                print(f"\nAttribute name: {attr.name}")
                # For simplicity, we only print a few key attributes.
                # The full tree structure is encoded here.
                if attr.name == "n_targets":
                    print(f"  Number of targets: {attr.i}")
                elif attr.name == "nodes_nodeids":
                    print(f"  Node feature ids: {attr.ints[:]}")
                elif attr.name == "nodes_hitrates":
                    print(f"  Node hit rates: {attr.floats[:]}")
                elif attr.name == "nodes_values":
                    print(f"  Nodes values: {attr.floats[:]}")
                elif attr.name == "target_weights":
                    print(f"  Target weights: {attr.floats[:]}")
                # You would need to parse these attributes to reconstruct the trees
    else:
        print("No TreeEnsembleRegressor node found.")
